In [1]:


from feat_engin_helpers import *
import numpy as np
import pandas as pd

Y, cases, unit_ids = load_uc()
long = to_long(Y, cases, unit_ids)

METRICS = ["correlation", "auc", "strength", "sign_consistency", "frac_missing"]
PLANT_TO_RES = {"Hogga": ["Bandak"], "Tokke": ["Vinjevatn"], "Lio": ["Byrtevatn"], "Byrte": ["Botnedalsvatn"],
                "Vinje": ["Vaamarvatn"], "Haukeli": ["Vatjern"], "Songa": ["Bitdalsvatn", "Songavatn"],
                "Kjela": ["Foersvatn"], "Vesle Kjela": ["Kjelavatn"]}
TO_RIVER1 = {"Hyljelihyl": ["b_Kjelavatn"], "Vinjevatn": ["b_Hyljelihyl"], "Bandak": ["b_Vinjevatn", "r_Vest_Vassdraget", "b_Byrtevatn"]}
TO_RIVER2 = {"Kjelavatn": ["b_Kjelavatn"], "Hyljelihyl": ["b_Hyljelihyl"], "Vinjevatn": ["b_Vinjevatn", "r_Vest_Vassdraget"], 
            "Byrtevatn": ["b_Byrtevatn", "r_Vest_Vassdraget"], "Bandak": ["b_Bandak"], "Vaamarvatn": ["r_Vest_Vassdraget"]}
UPSTR_GEN = {"Songa": ["Totak"], "Byrte": ["Byrtevatn"]}

features = []
collected_scores = pd.DataFrame()
unit_ids = list(long["unit"].cat.categories)

def build_unit_feature(feature, unit_ids, plant_to_res, res_to_cols=None, agg="sum"):
    out, used = {}, set()
    for uid in unit_ids:
        plant = uid.rsplit("_", 1)[0]
        reservoirs = plant_to_res.get(plant) or plant_to_res.get(plant.replace("_", " ")) or []
        wanted = reservoirs if res_to_cols is None else [c for r in reservoirs for c in res_to_cols.get(r, [])]
        wanted = [c for c in dict.fromkeys(wanted) if c in feature.columns]   # dedupe
        used.update(wanted)
        out[uid] = (getattr(feature[wanted], agg)(axis=1) if wanted
                    else pd.Series(np.nan, index=feature.index, dtype="float32"))
    unmapped = [c for c in feature.columns if c not in used and (feature[c] > 0).any()]
    if unmapped:
        print("Active constraints not mapped to any unit:", unmapped)
    return pd.DataFrame(out)

def printScores(show=False, store=False):
    global collected_scores
    sub = long[long["case_idx"] % 7 == 0]
    train = sub[sub["year"] <= 2021]
    n_units_total = train["unit"].nunique()

    all_scores = []
    for unit, g in train.groupby("unit", observed=True):
        s = filter_scores(g, features, "y")
        if s.empty:
            continue
        all_scores.append(s.assign(unit=unit))
    if not all_scores:
        print("No feature could be scored for any unit.")
        return

    per_unit = pd.concat(all_scores, ignore_index=True)
    per_unit["feature_unit"] = per_unit["feature"] + "_" + per_unit["unit"].astype(str)
    per_unit["is_avg"] = False

    # mean over the units where the feature was actually scored
    avg = (per_unit.groupby("feature")
           .agg(**{m: (m, "mean") for m in METRICS}, n_units=("unit", "nunique"))
           .reset_index())
    avg["n_units_total"] = n_units_total
    avg["feature_unit"] = avg["feature"] + "_AVG"
    avg["is_avg"] = True

    parts = [per_unit, avg] + ([collected_scores] if not collected_scores.empty else [])
    result = (pd.concat(parts, ignore_index=True)
              .drop_duplicates("feature_unit", keep="first")   # re-running replaces old rows
              .sort_values("strength", ascending=False)
              .reset_index(drop=True))
    collected_scores = result

    def highlight_avg(row):
        style = "background-color: #ffe08a; font-weight: bold" if row["is_avg"] else ""
        return [style] * len(row)

    if show:
        if show == "avg":
            view = result[result["is_avg"]]
            cols = ["feature_unit"] + METRICS + ["n_units", "n_units_total"]
        else:
            view = result
            cols = ["feature_unit"] + METRICS + ["n_units"]
        display(
            view[cols + ["is_avg"]]
            .style.apply(highlight_avg, axis=1)
            .format(precision=4, na_rep="")
            .hide(axis="columns", subset=["is_avg"])
            .hide(axis="index")
        )
    # SAVE RESULTS TO CSV
    if store:
        result[cols].to_csv("filter_scores.csv", index=False)

In [2]:
#2. Attach raw features
price = read_timeseries("../sintef_project/data/kernel/Historical_day_ahead_price_2015_2025.csv")["Day-ahead price (EUR/MWh)"]
attach_hourly(long, price, "price")
features.append("price")

printScores()

In [3]:
long = to_long(Y, cases, unit_ids)
features = []

inflow = read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv")
inflow = inflow.astype(np.float32)
inflow = inflow.loc['2015-01-01':]

df_units = build_unit_feature(inflow, unit_ids, PLANT_TO_RES)
long = attach_unit_specific(long, df_units, "inflow")
features.append("inflow")

printScores()

# SAVE RESULTS TO CSV
#result[cols].to_csv("filter_scores.csv", index=False)

Active constraints not mapped to any unit: ['Langesae', 'Staavatn', 'Bordalsvatn', 'Hyljelihyl', 'Venemo', 'Totak', 'Langeidvatn', 'r_Vest_Vassdraget']


In [4]:
long = to_long(Y, cases, unit_ids)
features = []

storage = read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv")
storage = storage.astype(np.float32)

df_units = build_unit_feature(storage, unit_ids, PLANT_TO_RES)
long = attach_unit_specific(long, df_units, "storage", "daily")
features.append("storage")

printScores()

Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']


In [5]:
long = to_long(Y, cases, unit_ids)
features = []

value = read_timeseries("../sintef_project/data/kernel/Synthetic_water_value_2015_2024.csv")
value = value.astype(np.float32)

df_units = build_unit_feature(value, unit_ids, PLANT_TO_RES)
long = attach_unit_specific(long, df_units, "value", "daily")
features.append("value")

printScores()

Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']


In [6]:
long = to_long(Y, cases, unit_ids)
features = []

min_flow = read_timeseries("../sintef_project/data/extended/Constraint_min_flow.csv")
min_flow = min_flow.astype(np.float32)

df_units = build_unit_feature(min_flow, unit_ids, PLANT_TO_RES, TO_RIVER1, agg="sum")
long = attach_unit_specific(long, df_units, "min_flow_upstr", "hourly")
features.append("min_flow_upstr")

df_units = build_unit_feature(min_flow, unit_ids, PLANT_TO_RES, TO_RIVER2, agg="sum")
long = attach_unit_specific(long, df_units, "min_flow_downstr", "hourly")
features.append("min_flow_downstr")

printScores()

Active constraints not mapped to any unit: ['b_Kjelavatn', 'b_Bandak']
Active constraints not mapped to any unit: ['b_Hyljelihyl']


In [7]:
long = to_long(Y, cases, unit_ids)
features = []

min_vol = read_timeseries("../sintef_project/data/extended/Constraint_min_volume.csv")
min_vol = min_vol.astype(np.float32)

df_units = build_unit_feature(min_vol, unit_ids, UPSTR_GEN)
long = attach_unit_specific(long, df_units, "min_vol_upstr", "hourly")
features.append("min_vol_upstr")

df_units = build_unit_feature(min_vol, unit_ids, PLANT_TO_RES)
long = attach_unit_specific(long, df_units, "min_vol_downstr", "hourly")
features.append("min_vol_downstr")

printScores()

Active constraints not mapped to any unit: ['Staavatn']
Active constraints not mapped to any unit: ['Totak', 'Staavatn']


In [8]:
# ROLLING AND LAGGED PRICES
#

long = to_long(Y, cases, unit_ids)
features = []

price = read_timeseries("../sintef_project/data/kernel/Historical_day_ahead_price_2015_2025.csv")["Day-ahead price (EUR/MWh)"]
price = price.astype(np.float32).asfreq("h")   # regular hourly grid so shift() means "hours"

derived = {"price": price}

# Lags (past) and leads (future prices within the week)
for k in [1, 2, 3, 6, 12, 24, 48, 168]:
    derived[f"price_lag{k}h"] = price.shift(k)
for k in [1, 3, 6, 12, 24]:
    derived[f"price_lead{k}h"] = price.shift(-k)

# Trailing rolling stats
for w in [6, 12, 24, 72, 168]:
    r = price.rolling(w, min_periods=w // 2)
    mean = r.mean()
    derived[f"price_rollmean{w}h"] = mean
    derived[f"price_rollstd{w}h"] = r.std()
    derived[f"price_rollmin{w}h"] = r.min()
    derived[f"price_rollmax{w}h"] = r.max()
    derived[f"price_dev_rollmean{w}h"] = price - mean        # price relative to its recent level
    derived[f"price_rank{w}h"] = r.rank(pct=True)            # where the price sits in its recent range

# Within-day shape
day = price.index.floor("D")
g = price.groupby(day)
derived["price_daymean"] = g.transform("mean")
derived["price_dev_daymean"] = price - derived["price_daymean"]
derived["price_dayrank"] = g.rank(pct=True)
derived["price_daymax_diff"] = price - g.transform("max")
derived["price_daymin_diff"] = price - g.transform("min")

for name, s in derived.items():
    attach_hourly(long, s, name)
    features.append(name)

printScores()

In [9]:
# CUM INFLOW AND DAY TO DAY CHANGES
#

long = to_long(Y, cases, unit_ids)
features = []

inflow = read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv")
inflow = inflow.astype(np.float32)          # keep full history so windows are warm at 2015-01-01

df_units = build_unit_feature(inflow, unit_ids, PLANT_TO_RES)

# Daily grid (mean = average flow rate; use .sum() instead if the values are volumes per hour)
daily = df_units.resample("D").mean()

LAG = 0   # set to 1 to use only inflow known up to the previous day

derived = {"inflow": daily}

# Cumulative inflow over trailing windows (days)
for w in [3, 7, 14, 30, 60, 90]:
    derived[f"inflow_cum{w}d"] = daily.rolling(w, min_periods=max(1, w // 2)).sum()

# Cumulative inflow since the start of the hydrological year (1 Sep)
wyear = daily.index.year + (daily.index.month >= 9)
derived["inflow_cum_wyear"] = daily.groupby(wyear).cumsum()

# Day-to-day changes
derived["inflow_diff1d"] = daily.diff(1)
derived["inflow_diff2d"] = daily.diff(2)
derived["inflow_diff3d"] = daily.diff(3)
derived["inflow_diff7d"] = daily.diff(7)
derived["inflow_pct1d"] = daily.pct_change(1, fill_method=None).replace([np.inf, -np.inf], np.nan)
derived["inflow_accel1d"] = daily.diff(1).diff(1)      # change in the change

# Current level relative to recent level
for w in [7, 30]:
    m = daily.rolling(w, min_periods=max(1, w // 2)).mean()
    derived[f"inflow_dev_mean{w}d"] = daily - m
    derived[f"inflow_ratio_mean{w}d"] = (daily / m).replace([np.inf, -np.inf], np.nan)

# Trend of the smoothed series
derived["inflow_trend7d"] = daily.rolling(7, min_periods=4).mean().diff(1)

for name, d in derived.items():
    d = d.shift(LAG).loc["2015-01-01":]
    long = attach_unit_specific(long, d, name, "daily")
    features.append(name)

printScores()

Active constraints not mapped to any unit: ['Langesae', 'Staavatn', 'Bordalsvatn', 'Hyljelihyl', 'Venemo', 'Totak', 'Langeidvatn', 'r_Vest_Vassdraget']


In [10]:
# UPSTREAM VOLUME AND INFLOW
#

UPSTREAM = {"Bandak": ["Vinjevatn", "Botnedalsvatn", "Byrtevatn"], "Byrtevatn": ["Botnedalsvatn"], 
                 "Vinjevatn": ["Vaamarvatn", "Vatjern", "Hyljelihyl"], "Vatjern": ["Langeidvatn"], "Vaamarvatn": ["Totak"], 
                 "Totak": ["Songavatn", "Bitdalsvatn", "Venemo"], "Venemo": ["Hyljelihyl", "Bordalsvatn"],
                 "Hyljelihyl": ["Bordalsvatn", "Foersvatn", "Kjelavatn"], "Foersvatn": ["Langesae", "Kjelavatn", "Bordalsvatn"], 
                 "Kjelavatn": ["Staavatn"]}

long = to_long(Y, cases, unit_ids)
features = []

def total_upstream(res, upstream):
    seen, stack = set(), list(upstream.get(res, []))
    while stack:
        r = stack.pop()
        if r in seen:
            continue
        seen.add(r)
        stack.extend(upstream.get(r, []))
    return seen

def plant_upstream_map(plant_to_res, upstream, total=False):
    out = {}
    for plant, reservoirs in plant_to_res.items():
        ups = set()
        for r in reservoirs:
            ups |= total_upstream(r, upstream) if total else set(upstream.get(r, []))
        ups -= set(reservoirs)
        out[plant] = sorted(ups)
    return out

UP_DIRECT = plant_upstream_map(PLANT_TO_RES, UPSTREAM, total=False)
UP_TOTAL = plant_upstream_map(PLANT_TO_RES, UPSTREAM, total=True)

# Upstream storage (daily)
storage = read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv").astype(np.float32)

df_units = build_unit_feature(storage, unit_ids, UP_DIRECT)
long = attach_unit_specific(long, df_units, "storage_up_direct", "daily")
features.append("storage_up_direct")

df_units = build_unit_feature(storage, unit_ids, UP_TOTAL)
long = attach_unit_specific(long, df_units, "storage_up_total", "daily")
features.append("storage_up_total")

# Upstream inflow
inflow = read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv").astype(np.float32)
inflow = inflow.loc['2015-01-01':]

df_units = build_unit_feature(inflow, unit_ids, UP_DIRECT)
long = attach_unit_specific(long, df_units, "inflow_up_direct")
features.append("inflow_up_direct")

df_units = build_unit_feature(inflow, unit_ids, UP_TOTAL)
long = attach_unit_specific(long, df_units, "inflow_up_total")
features.append("inflow_up_total")

printScores()

Active constraints not mapped to any unit: ['Foersvatn', 'Venemo', 'Bitdalsvatn', 'Songavatn', 'Bandak']
Active constraints not mapped to any unit: ['Bandak']
Active constraints not mapped to any unit: ['Foersvatn', 'Venemo', 'Bitdalsvatn', 'Songavatn', 'Bandak', 'r_Vest_Vassdraget']
Active constraints not mapped to any unit: ['Bandak', 'r_Vest_Vassdraget']


In [11]:
# DAY TO DAY CHANGES IN STORAGE
#

long = to_long(Y, cases, unit_ids)
features = []

storage = read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv").astype(np.float32)

sources = {
    "storage": PLANT_TO_RES,
    "storage_up_direct": UP_DIRECT,
    "storage_up_total": UP_TOTAL,
}

for base, mapping in sources.items():
    daily = build_unit_feature(storage, unit_ids, mapping).asfreq("D")

    derived = {}
    for k in [1, 2, 3, 7, 14, 30]:
        derived[f"{base}_diff{k}d"] = daily.diff(k)

    derived[f"{base}_pct1d"] = daily.pct_change(1, fill_method=None).replace([np.inf, -np.inf], np.nan)
    derived[f"{base}_pct7d"] = daily.pct_change(7, fill_method=None).replace([np.inf, -np.inf], np.nan)
    derived[f"{base}_accel1d"] = daily.diff(1).diff(1)
    derived[f"{base}_trend7d"] = daily.rolling(7, min_periods=4).mean().diff(1)

    for w in [7, 30]:
        derived[f"{base}_dev_mean{w}d"] = daily - daily.rolling(w, min_periods=max(1, w // 2)).mean()

    for name, d in derived.items():
        long = attach_unit_specific(long, d, name, "daily")
        features.append(name)

printScores()

Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']
Active constraints not mapped to any unit: ['Foersvatn', 'Venemo', 'Bitdalsvatn', 'Songavatn', 'Bandak']
Active constraints not mapped to any unit: ['Bandak']


In [12]:
# HOUR, DAY, MONTH, SEASON, WEEKEND
#

long = to_long(Y, cases, unit_ids)
features = []

ts = long["timestamp"].dt
hour = long["hour_of_day"].astype(np.float32)
dow = ts.dayofweek.astype(np.float32)          # 0 = Monday
month = ts.month.astype(np.float32)
doy = ts.dayofyear.astype(np.float32)

derived = {
    "hour_of_day": hour,
    "day_of_week": dow,
    "month": month,
    "day_of_year": doy,
    "is_weekend": (dow >= 5).astype(np.float32),
    "season": (month % 12 // 3).astype(np.float32),   # 0 winter, 1 spring, 2 summer, 3 autumn
}

# Cyclic encodings (raw values are monotone, so AUC misses wrap-around patterns)
for name, val, period in [("hour", hour, 24), ("dow", dow, 7), ("month", month - 1, 12), ("doy", doy - 1, 365.25)]:
    derived[f"{name}_sin"] = np.sin(2 * np.pi * val / period).astype(np.float32)
    derived[f"{name}_cos"] = np.cos(2 * np.pi * val / period).astype(np.float32)

# Indicators for non-monotone categorical effects
for s, label in enumerate(["winter", "spring", "summer", "autumn"]):
    derived[f"is_{label}"] = (derived["season"] == s).astype(np.float32)
for h in range(24):
    derived[f"is_hour{h}"] = (hour == h).astype(np.float32)
for d, label in enumerate(["mon", "tue", "wed", "thu", "fri", "sat", "sun"]):
    derived[f"is_{label}"] = (dow == d).astype(np.float32)
for m in range(1, 13):
    derived[f"is_month{m}"] = (month == m).astype(np.float32)

# Interaction: weekend x hour
derived["weekend_x_hour_sin"] = derived["is_weekend"] * derived["hour_sin"]
derived["weekend_x_hour_cos"] = derived["is_weekend"] * derived["hour_cos"]

for name, s in derived.items():
    long[name] = s.to_numpy()
    features.append(name)

printScores()

In [13]:
# PRICE MINUS VALUE
#

long = to_long(Y, cases, unit_ids)
features = []

price = read_timeseries("../sintef_project/data/kernel/Historical_day_ahead_price_2015_2025.csv")["Day-ahead price (EUR/MWh)"].astype(np.float32)
value = read_timeseries("../sintef_project/data/kernel/Synthetic_water_value_2015_2024.csv").astype(np.float32)

# mean (not sum) so plants with several reservoirs stay on the price scale
df_value = build_unit_feature(value, unit_ids, PLANT_TO_RES, agg="mean")
long = attach_unit_specific(long, df_value, "_value", "daily")
long = attach_hourly(long, price, "_price")
long = attach_daily(long, price.resample("D").mean(), "_price_daymean")

p, v, pd_ = long["_price"], long["_value"], long["_price_daymean"]

derived = {
    "price_minus_value": p - v,
    "price_over_value": (p / v).replace([np.inf, -np.inf], np.nan),
    "price_minus_value_sign": np.sign(p - v),
    "daymean_price_minus_value": pd_ - v,
    "daymean_price_over_value": (pd_ / v).replace([np.inf, -np.inf], np.nan),
}

for name, s in derived.items():
    long[name] = s.astype(np.float32).to_numpy()
    features.append(name)

long = long.drop(columns=["_value", "_price", "_price_daymean"])

printScores()

Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']


In [14]:
# ROLLING TRENDS AND SLOPES
#

long = to_long(Y, cases, unit_ids)
features = []

def roll_mean(d, w):
    return d.rolling(w, min_periods=max(1, w // 2)).mean()

def roll_slope(d, w):
    # least-squares slope per day over the trailing window
    x = np.arange(w, dtype=np.float32)
    x = x - x.mean()
    denom = (x ** 2).sum()
    return d.rolling(w, min_periods=w).apply(lambda y: np.dot(x, y - y.mean()) / denom, raw=True)

# ---------- Inflow trends ----------
inflow = read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv").astype(np.float32)

for base, mapping in {"inflow": PLANT_TO_RES,
                      "inflow_up_direct": UP_DIRECT,
                      "inflow_up_total": UP_TOTAL}.items():
    daily = build_unit_feature(inflow, unit_ids, mapping).resample("D").mean()

    derived = {}
    for short, longw in [(3, 7), (7, 30), (14, 60), (30, 90), (7, 90)]:
        ms, ml = roll_mean(daily, short), roll_mean(daily, longw)
        derived[f"{base}_{short}d_minus_{longw}d"] = ms - ml
        derived[f"{base}_{short}d_over_{longw}d"] = (ms / ml).replace([np.inf, -np.inf], np.nan)
    for w in [7, 14, 30]:
        derived[f"{base}_slope{w}d"] = roll_slope(daily, w)

    for name, d in derived.items():
        long = attach_unit_specific(long, d.loc["2015-01-01":], name, "daily")
        features.append(name)

# ---------- Storage rate of change ----------
storage = read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv").astype(np.float32)

for base, mapping in {"storage": PLANT_TO_RES,
                      "storage_up_direct": UP_DIRECT,
                      "storage_up_total": UP_TOTAL}.items():
    daily = build_unit_feature(storage, unit_ids, mapping).asfreq("D")

    derived = {}
    for k in [7, 14, 30]:
        derived[f"{base}_roc{k}d"] = (daily - daily.shift(k)) / k           # per-day rate
        derived[f"{base}_roc{k}d_rel"] = ((daily - daily.shift(k)) / daily.shift(k)).replace([np.inf, -np.inf], np.nan)
    for short, longw in [(3, 7), (7, 30), (14, 60)]:
        ms, ml = roll_mean(daily, short), roll_mean(daily, longw)
        derived[f"{base}_{short}d_minus_{longw}d"] = ms - ml
    for w in [7, 14, 30]:
        derived[f"{base}_slope{w}d"] = roll_slope(daily, w)

    for name, d in derived.items():
        long = attach_unit_specific(long, d, name, "daily")
        features.append(name)

printScores()

Active constraints not mapped to any unit: ['Langesae', 'Staavatn', 'Bordalsvatn', 'Hyljelihyl', 'Venemo', 'Totak', 'Langeidvatn', 'r_Vest_Vassdraget']
Active constraints not mapped to any unit: ['Foersvatn', 'Venemo', 'Bitdalsvatn', 'Songavatn', 'Bandak', 'r_Vest_Vassdraget']
Active constraints not mapped to any unit: ['Bandak', 'r_Vest_Vassdraget']
Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']
Active constraints not mapped to any unit: ['Foersvatn', 'Venemo', 'Bitdalsvatn', 'Songavatn', 'Bandak']
Active constraints not mapped to any unit: ['Bandak']


In [15]:
# DISTANCE TO MINVOL AND MINFLOW
#

long = to_long(Y, cases, unit_ids)
features = []

def add(name, s):
    global long
    long[name] = np.asarray(s, dtype=np.float32)
    features.append(name)

def rel(a, b):
    return (a / b).replace([np.inf, -np.inf], np.nan)

# ---------- Distance to min volume constraint ----------
storage = read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv").astype(np.float32)
min_vol = read_timeseries("../sintef_project/data/extended/Constraint_min_volume.csv").astype(np.float32)

for tag, mapping in {"upstr": UPSTR_GEN, "downstr": PLANT_TO_RES}.items():
    df_sto = build_unit_feature(storage, unit_ids, mapping)
    df_min = build_unit_feature(min_vol, unit_ids, mapping)

    long = attach_unit_specific(long, df_sto, "_sto", "daily")
    long = attach_unit_specific(long, df_min, "_minvol", "hourly")

    sto, mv = long["_sto"], long["_minvol"]
    dist = sto - mv
    active = mv > 0

    add(f"dist_min_vol_{tag}", dist)
    add(f"dist_min_vol_{tag}_rel", rel(dist, sto))
    add(f"dist_min_vol_{tag}_ratio", rel(sto, mv))
    add(f"dist_min_vol_{tag}_active_only", dist.where(active))
    add(f"min_vol_{tag}_is_active", active.astype(np.float32).where(mv.notna()))
    add(f"min_vol_{tag}_is_binding", (dist <= 0).astype(np.float32).where(dist.notna()))

    long = long.drop(columns=["_sto", "_minvol"])

# ---------- Distance to min flow constraint ----------
# No observed release series, so reservoir inflow is used as the available flow
inflow = read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv").astype(np.float32)
inflow = inflow.loc["2015-01-01":]
min_flow = read_timeseries("../sintef_project/data/extended/Constraint_min_flow.csv").astype(np.float32)

df_inf = build_unit_feature(inflow, unit_ids, PLANT_TO_RES)
long = attach_unit_specific(long, df_inf, "_inflow", "hourly")

for tag, river_map in {"upstr": TO_RIVER1, "downstr": TO_RIVER2}.items():
    df_mf = build_unit_feature(min_flow, unit_ids, PLANT_TO_RES, river_map, agg="sum")
    long = attach_unit_specific(long, df_mf, "_minflow", "hourly")

    inf, mf = long["_inflow"], long["_minflow"]
    dist = inf - mf
    active = mf > 0

    add(f"dist_min_flow_{tag}", dist)
    add(f"dist_min_flow_{tag}_ratio", rel(inf, mf))
    add(f"dist_min_flow_{tag}_active_only", dist.where(active))
    add(f"min_flow_{tag}_is_active", active.astype(np.float32).where(mf.notna()))
    add(f"min_flow_{tag}_is_binding", (dist <= 0).astype(np.float32).where(dist.notna() & active))

    long = long.drop(columns=["_minflow"])

long = long.drop(columns=["_inflow"])

printScores()

Active constraints not mapped to any unit: ['Kjelavatn', 'Staavatn', 'Langesae', 'Foersvatn', 'Venemo', 'Botnedalsvatn', 'Bitdalsvatn', 'Vaamarvatn', 'Langeidvatn', 'Songavatn', 'Vinjevatn', 'Bandak', 'Bordalsvatn', 'Hyljelihyl', 'Vatjern']
Active constraints not mapped to any unit: ['Staavatn']
Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']
Active constraints not mapped to any unit: ['Totak', 'Staavatn']
Active constraints not mapped to any unit: ['Langesae', 'Staavatn', 'Bordalsvatn', 'Hyljelihyl', 'Venemo', 'Totak', 'Langeidvatn', 'r_Vest_Vassdraget']
Active constraints not mapped to any unit: ['b_Kjelavatn', 'b_Bandak']
Active constraints not mapped to any unit: ['b_Hyljelihyl']


In [16]:
long = to_long(Y, cases, unit_ids)
features = []

def add(name, s):
    long[name] = np.asarray(s, dtype=np.float32)
    features.append(name)

def rel(a, b):
    return (a / b).replace([np.inf, -np.inf], np.nan)

def anomaly(daily):
    # deviation from the day-of-year mean over the full history
    clim = daily.groupby(daily.index.dayofyear).transform("mean")
    return daily - clim, rel(daily, clim)

# ---------- Seasonal anomalies ----------
sources = {
    "storage": (read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv"), "daily"),
    "inflow": (read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv"), "daily"),
    "value": (read_timeseries("../sintef_project/data/kernel/Synthetic_water_value_2015_2024.csv"), "daily"),
}
for base, (raw, _) in sources.items():
    raw = raw.astype(np.float32)
    agg = "mean" if base == "value" else "sum"
    daily = build_unit_feature(raw, unit_ids, PLANT_TO_RES, agg=agg).resample("D").mean()
    a_abs, a_rel = anomaly(daily)
    for name, d in {f"{base}_anom": a_abs, f"{base}_anom_rel": a_rel}.items():
        long = attach_unit_specific(long, d.loc["2015-01-01":], name, "daily")
        features.append(name)

    if base == "inflow":
        cum30 = daily.rolling(30, min_periods=15).sum()
        a_abs, a_rel = anomaly(cum30)
        for name, d in {"inflow_cum30d_anom": a_abs, "inflow_cum30d_anom_rel": a_rel}.items():
            long = attach_unit_specific(long, d.loc["2015-01-01":], name, "daily")
            features.append(name)

# ---------- Water value dynamics ----------
value = sources["value"][0].astype(np.float32)
df_val = build_unit_feature(value, unit_ids, PLANT_TO_RES, agg="mean").asfreq("D")
for k in [1, 7, 30]:
    long = attach_unit_specific(long, df_val.diff(k), f"value_diff{k}d", "daily")
    features.append(f"value_diff{k}d")

# ---------- Price: case-relative shape, ramps, peaks ----------
price = read_timeseries("../sintef_project/data/kernel/Historical_day_ahead_price_2015_2025.csv")["Day-ahead price (EUR/MWh)"]
price = price.astype(np.float32).asfreq("h")

long = attach_hourly(long, price, "_p")
long = attach_hourly(long, price.shift(1), "_p_prev")
long = attach_hourly(long, price.shift(-1), "_p_next")
long = attach_hourly(long, price.shift(3), "_p_prev3")

p = long["_p"]
g = long.groupby(["case_idx", "unit"], observed=True)["_p"]   # one 168h path per case (same for all units)
case_mean, case_std = g.transform("mean"), g.transform("std")
case_max, case_min = g.transform("max"), g.transform("min")
case_rank = g.rank(pct=True)

add("price_case_rank", case_rank)
add("price_case_dev_mean", p - case_mean)
add("price_case_z", rel(p - case_mean, case_std))
add("price_case_range_pos", rel(p - case_min, case_max - case_min))
add("price_case_mean", case_mean)
add("price_case_std", case_std)
add("price_case_is_top10", (case_rank >= 0.9).astype(np.float32).where(p.notna()))
add("price_case_is_bottom10", (case_rank <= 0.1).astype(np.float32).where(p.notna()))

add("price_ramp1h", p - long["_p_prev"])
add("price_ramp3h", p - long["_p_prev3"])
add("price_ramp_next1h", long["_p_next"] - p)
add("price_is_local_max", ((p > long["_p_prev"]) & (p > long["_p_next"])).astype(np.float32).where(p.notna()))
add("price_is_local_min", ((p < long["_p_prev"]) & (p < long["_p_next"])).astype(np.float32).where(p.notna()))

# Peak-to-trough spread of the day and the hour's position in it
day_key = long["timestamp"].dt.floor("D")
gd = long.groupby(["case_idx", "unit", day_key], observed=True)["_p"]
d_max, d_min = gd.transform("max"), gd.transform("min")
add("price_day_spread", d_max - d_min)
add("price_day_range_pos", rel(p - d_min, d_max - d_min))

long = long.drop(columns=["_p", "_p_prev", "_p_next", "_p_prev3"])

printScores(True, True)

Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']
Active constraints not mapped to any unit: ['Langesae', 'Staavatn', 'Bordalsvatn', 'Hyljelihyl', 'Venemo', 'Totak', 'Langeidvatn', 'r_Vest_Vassdraget']
Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']
Active constraints not mapped to any unit: ['Staavatn', 'Langesae', 'Venemo', 'Langeidvatn', 'Totak', 'Bordalsvatn', 'Hyljelihyl']


feature_unit,correlation,auc,strength,sign_consistency,frac_missing,n_units
daymean_price_over_value_Vesle Kjela_G1,0.7861,0.9540,0.4540,1.0000,0.0000,
price_over_value_Vesle Kjela_G1,0.7812,0.9512,0.4512,1.0000,0.0000,
price_minus_value_Byrte_G1,0.7698,0.9474,0.4474,1.0000,0.0000,
price_minus_value_Vesle Kjela_G1,0.7711,0.9453,0.4453,1.0000,0.0000,
daymean_price_minus_value_Vesle Kjela_G1,0.7708,0.9452,0.4452,1.0000,0.0000,
price_minus_value_Songa_G1,0.7668,0.9446,0.4446,1.0000,0.0000,
daymean_price_minus_value_Byrte_G1,0.7642,0.9442,0.4442,1.0000,0.0000,
price_over_value_Songa_G1,0.7647,0.9433,0.4433,1.0000,0.0000,
daymean_price_over_value_Byrte_G1,0.7595,0.9415,0.4415,1.0000,0.0000,
price_over_value_Byrte_G1,0.7586,0.9409,0.4409,1.0000,0.0000,


In [17]:
printScores("avg")

feature_unit,correlation,auc,strength,sign_consistency,frac_missing,n_units,n_units_total
dist_min_vol_downstr_ratio_AVG,0.5576,0.8454,0.3454,0.6000,0.8743,1.0000,14.0000
dist_min_vol_downstr_active_only_AVG,0.5576,0.8454,0.3454,0.6000,0.8743,1.0000,14.0000
price_over_value_AVG,0.4720,0.7839,0.2839,1.0000,0.0000,14.0000,14.0000
price_minus_value_AVG,0.4629,0.7784,0.2784,1.0000,0.0000,14.0000,14.0000
daymean_price_over_value_AVG,0.4621,0.7778,0.2778,1.0000,0.0000,14.0000,14.0000
daymean_price_minus_value_AVG,0.4509,0.7711,0.2711,1.0000,0.0000,14.0000,14.0000
storage_anom_AVG,0.4127,0.7484,0.2484,0.9780,0.0000,13.0000,14.0000
storage_anom_rel_AVG,0.4073,0.7451,0.2451,0.9780,0.0000,13.0000,14.0000
price_minus_value_sign_AVG,0.4491,0.7322,0.2322,0.9898,0.0000,14.0000,14.0000
value_AVG,-0.3440,0.2927,0.2073,0.9898,0.0000,14.0000,14.0000
